In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
print(torch.cuda.is_available())


True


In [ ]:
!pip install darts yfinance --quiet

In [ ]:
!pip install darts yfinance pytorch-lightning --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch

from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler

print("GPU Available:", torch.cuda.is_available())


GPU Available: True


In [ ]:
nse_stocks = [
"RELIANCE.NS","TCS.NS","INFY.NS","HDFCBANK.NS","ICICIBANK.NS",
"LT.NS","ITC.NS","SBIN.NS","AXISBANK.NS","KOTAKBANK.NS",
"BHARTIARTL.NS","HINDUNILVR.NS","ASIANPAINT.NS","BAJFINANCE.NS",
"MARUTI.NS","HCLTECH.NS","SUNPHARMA.NS","TITAN.NS","WIPRO.NS",
"ULTRACEMCO.NS","ONGC.NS","NTPC.NS","POWERGRID.NS","TATASTEEL.NS",
"TATAMOTORS.NS","ADANIENT.NS","ADANIPORTS.NS","COALINDIA.NS",
"INDUSINDBK.NS","DRREDDY.NS","TECHM.NS","JSWSTEEL.NS","GRASIM.NS",
"HINDALCO.NS","DIVISLAB.NS","EICHERMOT.NS","BAJAJFINSV.NS",
"BRITANNIA.NS","CIPLA.NS","HEROMOTOCO.NS","NESTLEIND.NS",
"SHREECEM.NS","SBILIFE.NS","HDFCLIFE.NS","TATACONSUM.NS",
"APOLLOHOSP.NS","UPL.NS","M&M.NS","BPCL.NS","IOC.NS",
"PIDILITIND.NS","GODREJCP.NS","DABUR.NS","AMBUJACEM.NS",
"ICICIPRULI.NS","BAJAJ-AUTO.NS","VEDL.NS","SIEMENS.NS",
"ADANIGREEN.NS","ADANIPOWER.NS","BANKBARODA.NS","PNB.NS",
"CANBK.NS","IDFCFIRSTB.NS","TORNTPHARM.NS","LUPIN.NS",
"COLPAL.NS","NAUKRI.NS","DMART.NS","HAVELLS.NS",
"CHOLAFIN.NS","MUTHOOTFIN.NS","ICICIGI.NS","BEL.NS",
"HAL.NS","INDIGO.NS","TVSMOTOR.NS","BOSCHLTD.NS",
"ABB.NS","SRF.NS","MPHASIS.NS","BANDHANBNK.NS",
"ASHOKLEY.NS","PAGEIND.NS","GLAND.NS","POLYCAB.NS",
"JINDALSTEL.NS","ACC.NS","BIOCON.NS","TRENT.NS"
]



In [ ]:
bse_stocks = [
"RELIANCE.BO","TCS.BO","INFY.BO","HDFCBANK.BO","ICICIBANK.BO",
"LT.BO","ITC.BO","SBIN.BO","AXISBANK.BO","KOTAKBANK.BO",
"BHARTIARTL.BO","HINDUNILVR.BO","ASIANPAINT.BO","BAJFINANCE.BO",
"MARUTI.BO","HCLTECH.BO","SUNPHARMA.BO","TITAN.BO","WIPRO.BO",
"ULTRACEMCO.BO","ONGC.BO","NTPC.BO","POWERGRID.BO","TATASTEEL.BO",
"TATAMOTORS.BO","COALINDIA.BO","INDUSINDBK.BO","DRREDDY.BO",
"TECHM.BO","JSWSTEEL.BO","GRASIM.BO","HINDALCO.BO",
"DIVISLAB.BO","EICHERMOT.BO","BAJAJFINSV.BO","BRITANNIA.BO",
"CIPLA.BO","HEROMOTOCO.BO","NESTLEIND.BO","SHREECEM.BO",
"SBILIFE.BO","HDFCLIFE.BO","TATACONSUM.BO","APOLLOHOSP.BO",
"UPL.BO","M&M.BO","BPCL.BO","IOC.BO"
]


In [ ]:
all_symbols = nse_stocks + bse_stocks
print("Total stocks:", len(all_symbols))


Total stocks: 138


In [ ]:
dataframes = []

for symbol in all_symbols:
    try:
        df = yf.download(symbol, start="2000-01-01", progress=False)

        if len(df) > 3500:  # ~15+ years filter
            df = df.reset_index()
            df["close"] = df["Close"]
            dataframes.append(df[["Date", "close"]])
            print("Added:", symbol)
        else:
            print("Skipped (short history):", symbol)

    except:
        print("Error:", symbol)

print("Final usable stocks:", len(dataframes))


/tmp/ipython-input-816438004.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2000-01-01", progress=False)


Added: RELIANCE.NS


/tmp/ipython-input-816438004.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2000-01-01", progress=False)


Added: TCS.NS


/tmp/ipython-input-816438004.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2000-01-01", progress=False)


...

Added: M&M.BO


/tmp/ipython-input-816438004.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2000-01-01", progress=False)


Added: BPCL.BO


/tmp/ipython-input-816438004.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2000-01-01", progress=False)


Added: IOC.BO
Final usable stocks: 122


In [ ]:
def create_return_series(df):
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    df = df.dropna()
    return TimeSeries.from_values(df["log_return"].values)


In [ ]:
series_list = []

for df in dataframes:
    ts = create_return_series(df)
    scaler = Scaler()
    ts_scaled = scaler.fit_transform(ts)
    series_list.append(ts_scaled)

print("Total Series Used:", len(series_list))


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Total Series Used: 122


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
model = TFTModel(
    input_chunk_length=90,
    output_chunk_length=30,
    hidden_size=48,      # slightly smaller
    lstm_layers=2,
    num_attention_heads=4,
    dropout=0.1,
    batch_size=64,       # bigger batch = faster
    n_epochs=30,         # reduced
    random_state=42,
    add_relative_index=True
)



model.fit(series_list, verbose=True)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                              ┃ Type                             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ train_metrics                     │ MetricCollection                 │      0 │ train │     0 │
│ 1  │ val_metrics                       │ MetricCollection                 │      0 │ train │     0 │
│ 2  │ input_embeddings                  │ _MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ static_covariates_vsn             │ _VariableSelectionNetwork        │      0 │ train │     0 │
│ 4  │ encoder_vsn                       │ _VariableSelectionNetwork        │  2.7 K │ train │     0 │
│ 5  │ decoder_vsn                       │ _VariableSelectionNetwork        │  1.3 K │ train │     0 │
│ 6  │ static_context_grn                │ _GatedResidualNetwork            │  9.5 K │ train │     0 │
│ 7  │ static_context_hidden_encoder_grn │ _GatedResidualNetwork            │  9.5 K │ train │     0 │
│ 8  │ static_context_cell_encoder_grn   │ _GatedResidualNetwork            │  9.5 K │ train │     0 │
│ 9  │ static_context_enrichment         │ _GatedResidualNetwork            │  9.5 K │ train │     0 │
│ 10 │ lstm_encoder                      │ LSTM                             │ 37.6 K │ train │     0 │
│ 11 │ lstm_decoder                      │ LSTM                             │ 37.6 K │ train │     0 │
│ 12 │ post_lstm_gan                     │ _GateAddNorm                     │  4.8 K │ train │     0 │
│ 13 │ static_enrichment_grn             │ _GatedResidualNetwork            │ 11.8 K │ train │     0 │
│ 14 │ multihead_attn                    │ _InterpretableMultiHeadAttention │  5.9 K │ train │     0 │
│ 15 │ post_attn_gan                     │ _GateAddNorm                     │  4.8 K │ train │     0 │
│ 16 │ feed_forward_block                │ _GatedResidualNetwork            │  9.5 K │ train │     0 │
│ 17 │ pre_output_gan                    │ _GateAddNorm                     │  4.8 K │ train │     0 │
│ 18 │ output_layer                      │ Linear                           │    833 │ train │     0 │
└────┴───────────────────────────────────┴──────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 159 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 159 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 172                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


TFTModel(output_chunk_shift=0, hidden_size=48, lstm_layers=2, num_attention_heads=4, full_attention=False, feed_forward=GatedResidualNetwork, dropout=0.1, hidden_continuous_size=8, categorical_embedding_sizes=None, add_relative_index=True, skip_interpolation=False, loss_fn=None, likelihood=None, norm_type=LayerNorm, use_static_covariates=True, input_chunk_length=90, output_chunk_length=30, batch_size=64, n_epochs=30, random_state=42)

In [ ]:
model.save("/content/drive/MyDrive/global_return_model")


In [ ]:
!zip -r global_model.zip /content/drive/MyDrive/global_return_model*


updating: content/drive/MyDrive/global_return_model (deflated 51%)
updating: content/drive/MyDrive/global_return_model.ckpt (deflated 12%)


In [ ]:
from google.colab import files
files.download("global_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r global_model.zip global_return_model*


	zip warning: name not matched: global_return_model*

zip error: Nothing to do! (try: zip -r global_model.zip . -i global_return_model*)


In [ ]:
from google.colab import files
files.download("global_return_model.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>